# Final Model Comparison Notebook

This notebook loads saved experiment outputs from Random Forest, XGBoost, and RUSBoost, then compares the models across:

- Step 4 thresholding
- Step 5 calibration
- Step 6 conformal prediction
- Step 7 cost-aware deferral tuning
- Step 9 final held-out evaluation
- Raw vs processed performance
- Safety metrics
- Cost-aware decision quality
- Coverage vs deferral trade-off

**Before running:** update the three experiment paths in the next cell.

In [1]:
# =========================================================
# 1. Import Dependencies
# =========================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## 2. Set Experiment Folder Paths

Update these paths to the actual timestamped folders you created for each model.

In [2]:
# =========================================================
# 2. Experiment Paths
# =========================================================

EXPERIMENTS = {
    "Random Forest": "experiments_rf/run_YYYY-MM-DD_HH-MM-SS",
    "XGBoost": "experiments_xgboost/run_YYYY-MM-DD_HH-MM-SS",
    "RUSBoost": "experiments_rusboost/run_YYYY-MM-DD_HH-MM-SS",
}

for model_name, path in EXPERIMENTS.items():
    print(model_name, "->", path, "| exists:", os.path.exists(path))

Random Forest -> experiments_rf/run_YYYY-MM-DD_HH-MM-SS | exists: False
XGBoost -> experiments_xgboost/run_YYYY-MM-DD_HH-MM-SS | exists: False
RUSBoost -> experiments_rusboost/run_YYYY-MM-DD_HH-MM-SS | exists: False


## 3. Helper Functions

In [3]:
# =========================================================
# 3. Helper Functions
# =========================================================

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def find_file(run_dir, candidates):
    """Return the first matching file from a list of possible filenames."""
    run_dir = Path(run_dir)
    for name in candidates:
        p = run_dir / name
        if p.exists():
            return p
    return None


def cm_to_array(cm):
    arr = np.array(cm)
    if arr.shape == (2, 2):
        return arr
    return np.array([[0, 0], [0, 0]])


def get_binary_counts_from_cm(cm):
    cm = cm_to_array(cm)
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)


def safe_get(d, keys, default=np.nan):
    cur = d
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur


def extract_model_summary(model_name, run_dir):
    run_dir = Path(run_dir)
    
    full_results_path = find_file(run_dir, [
        "full_pipeline_results.json",
        "pipeline_summary.json",
        "summary_json.json",
    ])
    
    if full_results_path is None:
        raise FileNotFoundError(f"No full results JSON found in {run_dir}")
    
    results = load_json(full_results_path)
    
    step4 = results.get("step4_thresholding", {})
    step5 = results.get("step5_calibration", {})
    step6 = results.get("step6_conformal_calibration", {})
    step7 = results.get("step7_deferral_tuning", {})
    step9 = results.get("step9_final_evaluation", {})
    
    overall = step9.get("overall_test", {})
    selective = step9.get("selective_prediction_test", {})
    conformal = step9.get("conformal_test", {})
    
    # Raw model = calibrated classifier prediction using Step 4 threshold, before deferral.
    raw_cm = cm_to_array(overall.get("confusion_matrix", [[0, 0], [0, 0]]))
    raw_tn, raw_fp, raw_fn, raw_tp = get_binary_counts_from_cm(raw_cm)
    raw_recall = raw_tp / max(raw_tp + raw_fn, 1)
    raw_precision = raw_tp / max(raw_tp + raw_fp, 1)
    raw_f1 = 2 * raw_precision * raw_recall / max(raw_precision + raw_recall, 1e-12)
    
    # Processed model = conformal + cost-aware deferral policy.
    processed_fn = selective.get("fn", np.nan)
    processed_fp = selective.get("fp", np.nan)
    processed_tp = selective.get("tp", np.nan)
    processed_tn = selective.get("tn", np.nan)
    processed_recall = selective.get("retained_recall", np.nan)
    processed_precision = selective.get("retained_precision", np.nan)
    
    return {
        "model": model_name,
        "run_dir": str(run_dir),
        
        # Step 4
        "step4_threshold": step4.get("base_threshold", overall.get("base_threshold", np.nan)),
        "step4_target_recall": step4.get("target_recall", np.nan),
        
        # Step 5
        "brier_score": step5.get("brier_score", np.nan),
        "log_loss": step5.get("log_loss", np.nan),
        "certainty_mean": step5.get("certainty_mean", np.nan),
        "certainty_median": step5.get("certainty_median", np.nan),
        
        # Step 6
        "conformal_coverage_cal": step6.get("coverage_cal", np.nan),
        "conformal_avg_set_size_cal": step6.get("avg_set_size_cal", np.nan),
        "conformal_ambiguous_rate_cal": step6.get("ambiguous_rate_cal", np.nan),
        "conformal_coverage_test": conformal.get("coverage_test", np.nan),
        "conformal_avg_set_size_test": conformal.get("avg_set_size_test", np.nan),
        "conformal_ambiguous_rate_test": conformal.get("ambiguous_rate_test", np.nan),
        
        # Step 7
        "best_tau": step7.get("best_tau", selective.get("tau", np.nan)),
        "best_cost_per_sample_cal": step7.get("best_cost_per_sample", np.nan),
        "coverage_at_best_tau_cal": step7.get("coverage_at_best_tau", np.nan),
        "deferred_fraction_at_best_tau_cal": step7.get("deferred_fraction_at_best_tau", np.nan),
        
        # Raw Step 9 performance
        "raw_roc_auc": overall.get("roc_auc", np.nan),
        "raw_pr_auc": overall.get("pr_auc", np.nan),
        "raw_recall": raw_recall,
        "raw_precision": raw_precision,
        "raw_f1": raw_f1,
        "raw_fn": raw_fn,
        "raw_fp": raw_fp,
        "raw_tp": raw_tp,
        "raw_tn": raw_tn,
        
        # Processed Step 9 selective prediction
        "processed_cost_per_sample": selective.get("cost_per_sample", np.nan),
        "processed_total_cost": selective.get("total_cost", np.nan),
        "processed_coverage": selective.get("coverage", np.nan),
        "processed_deferred_fraction": selective.get("deferred_fraction", np.nan),
        "processed_deferred_count": selective.get("deferred_count", np.nan),
        "processed_retained_error": selective.get("retained_error", np.nan),
        "processed_recall": processed_recall,
        "processed_precision": processed_precision,
        "processed_fn": processed_fn,
        "processed_fp": processed_fp,
        "processed_tp": processed_tp,
        "processed_tn": processed_tn,
    }


def load_optional_csv(run_dir, filename):
    p = Path(run_dir) / filename
    if p.exists():
        return pd.read_csv(p)
    return None

## 4. Load Saved Results for the Three Models

In [4]:
# =========================================================
# 4. Load Saved Results
# =========================================================

summaries = []
deferral_curves = {}
test_predictions = {}

for model_name, run_dir in EXPERIMENTS.items():
    print(f"Loading {model_name} from {run_dir}")
    summary = extract_model_summary(model_name, run_dir)
    summaries.append(summary)
    
    deferral_df = load_optional_csv(run_dir, "step7_deferral_search.csv")
    if deferral_df is not None:
        deferral_df["model"] = model_name
        deferral_curves[model_name] = deferral_df
    
    test_df = load_optional_csv(run_dir, "step9_test_predictions.csv")
    if test_df is not None:
        test_df["model"] = model_name
        test_predictions[model_name] = test_df

comparison_df = pd.DataFrame(summaries)
comparison_df

Loading Random Forest from experiments_rf/run_YYYY-MM-DD_HH-MM-SS


FileNotFoundError: No full results JSON found in experiments_rf/run_YYYY-MM-DD_HH-MM-SS

## 5. Step-by-Step Comparison Table

In [ ]:
# Step 4 to Step 9 summary columns
step_summary_cols = [
    "model",
    "step4_threshold",
    "brier_score",
    "log_loss",
    "conformal_coverage_cal",
    "conformal_coverage_test",
    "best_tau",
    "best_cost_per_sample_cal",
    "processed_cost_per_sample",
    "processed_coverage",
    "processed_deferred_fraction",
]

step_comparison = comparison_df[step_summary_cols].copy()
step_comparison

## 6. Performance Comparison: Raw vs Processed Models

**Raw model:** calibrated classifier using the Step 4 classification threshold, before conformal prediction and deferral.

**Processed model:** final decision pipeline after conformal prediction and cost-aware deferral. Since deferred cases are not automatically classified, ROC-AUC/PR-AUC remain raw probability-ranking metrics; processed evaluation focuses on retained recall, retained precision, FN, coverage, deferral rate, and cost.

In [ ]:
performance_cols = [
    "model",
    "raw_roc_auc",
    "raw_pr_auc",
    "raw_recall",
    "raw_precision",
    "raw_f1",
    "processed_recall",
    "processed_precision",
    "processed_retained_error",
    "processed_coverage",
    "processed_deferred_fraction",
]

performance_df = comparison_df[performance_cols].copy()
performance_df

In [ ]:
# Plot ROC-AUC and PR-AUC for raw models
ax = comparison_df.set_index("model")[["raw_roc_auc", "raw_pr_auc"]].plot(kind="bar", figsize=(8, 5))
plt.title("Raw Model Ranking Performance")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 7. Safety Comparison: Recall and False Negatives

In [ ]:
safety_cols = [
    "model",
    "raw_recall",
    "raw_fn",
    "processed_recall",
    "processed_fn",
    "processed_deferred_count",
    "processed_deferred_fraction",
    "processed_cost_per_sample",
]

safety_df = comparison_df[safety_cols].copy()
safety_df

In [ ]:
# Recall comparison: raw vs processed retained predictions
ax = comparison_df.set_index("model")[["raw_recall", "processed_recall"]].plot(kind="bar", figsize=(8, 5))
plt.title("Safety Comparison: Failure Recall")
plt.ylabel("Recall")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# False negative comparison: raw vs processed retained predictions
ax = comparison_df.set_index("model")[["raw_fn", "processed_fn"]].plot(kind="bar", figsize=(8, 5))
plt.title("Safety Comparison: False Negatives")
plt.ylabel("Number of False Negatives")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 8. Cost-Aware Decision Comparison

In [ ]:
cost_cols = [
    "model",
    "best_cost_per_sample_cal",
    "processed_cost_per_sample",
    "processed_total_cost",
    "processed_coverage",
    "processed_deferred_fraction",
]

cost_df = comparison_df[cost_cols].copy().sort_values("processed_cost_per_sample")
cost_df

In [ ]:
ax = comparison_df.set_index("model")[["processed_cost_per_sample"]].plot(kind="bar", figsize=(8, 5), legend=False)
plt.title("Final Cost-Aware Decision Quality")
plt.ylabel("Cost per Sample")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 9. Coverage vs Deferral Trade-off

In [ ]:
if deferral_curves:
    all_deferral_df = pd.concat(deferral_curves.values(), ignore_index=True)
    display(all_deferral_df.head())
else:
    all_deferral_df = pd.DataFrame()
    print("No step7_deferral_search.csv files found.")

In [ ]:
if not all_deferral_df.empty:
    plt.figure(figsize=(8, 5))
    for model_name, df in deferral_curves.items():
        plt.plot(df["coverage"], df["cost_per_sample"], marker="o", label=model_name)
    
    plt.title("Coverage vs Cost Trade-off")
    plt.xlabel("Coverage: Fraction Automatically Predicted")
    plt.ylabel("Cost per Sample")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
if not all_deferral_df.empty:
    plt.figure(figsize=(8, 5))
    for model_name, df in deferral_curves.items():
        plt.plot(df["tau"], df["deferred_fraction"], marker="o", label=model_name)
    
    plt.title("Deferral Threshold vs Deferred Fraction")
    plt.xlabel("Deferral Threshold Tau")
    plt.ylabel("Deferred Fraction")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## 10. Final Model Ranking

This ranking prioritizes safety and operational decision quality:

1. Lower processed cost per sample is better.
2. Lower processed false negatives are better.
3. Higher processed recall is better.
4. Lower deferred fraction is better, if safety is not reduced.

In [ ]:
ranking_df = comparison_df[[
    "model",
    "processed_cost_per_sample",
    "processed_fn",
    "processed_recall",
    "processed_coverage",
    "processed_deferred_fraction",
    "raw_pr_auc",
    "raw_roc_auc",
]].copy()

ranking_df = ranking_df.sort_values(
    by=["processed_cost_per_sample", "processed_fn", "processed_recall", "processed_deferred_fraction"],
    ascending=[True, True, False, True]
)

ranking_df

## 11. Save Final Comparison Outputs

In [ ]:
output_dir = Path("final_model_comparison_outputs")
output_dir.mkdir(exist_ok=True)

comparison_df.to_csv(output_dir / "full_model_comparison.csv", index=False)
step_comparison.to_csv(output_dir / "stepwise_comparison.csv", index=False)
performance_df.to_csv(output_dir / "performance_comparison.csv", index=False)
safety_df.to_csv(output_dir / "safety_comparison.csv", index=False)
cost_df.to_csv(output_dir / "cost_comparison.csv", index=False)
ranking_df.to_csv(output_dir / "final_model_ranking.csv", index=False)

if not all_deferral_df.empty:
    all_deferral_df.to_csv(output_dir / "all_models_deferral_curves.csv", index=False)

print(f"Saved comparison outputs to: {output_dir.resolve()}")